# small\_fable — reasoning traces → SFT → GRPO on Colab (compositional plans)

Joint planner+executor on your **hard reasoning traces**. The planner emits a **factored plan** —
a flat sequence of primitives + parameter-atoms (`MODEL as=truth_table … FINALIZE form=yes_no END`)
from one autoregressive head — and the executor **reasons in prose, then commits a `FINAL ANSWER`**.

- **Train on sets 1000 + 2000** (base + one-step-deeper).
- **Hold out set 3000 (flipped-answer) entirely** — the unseen generalization test: same problem
  shapes, but the correct answer flips, so only *reasoning* (not memorizing) survives it.
- All stages checkpoint to HF every ~10 min and `--resume`, so a Colab crash just needs a re-run.

> **Before running:** T4 GPU + a Hugging Face **write** token as a Colab secret `HF_TOKEN`. Then
> `Runtime → Run all`. If Colab dies, Run all again — finished stages skip, the dead one resumes.


## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — Runtime → Change runtime type → T4 GPU, then re-run.'


## 1 · Clone the repo & install deps

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt huggingface_hub
!pip uninstall -y torchao >/dev/null 2>&1; echo 'torchao removed (peft/torchao clash workaround)'
print('\nReady.')


## 2 · Hugging Face setup

In [ ]:
import os
from huggingface_hub import HfApi, create_repo, snapshot_download, login
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from huggingface_hub import notebook_login; notebook_login()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False); os.environ['HF_TOKEN'] = HF_TOKEN
api = HfApi()
HF_REPO = f"{api.whoami()['name']}/small_fable-planner"
create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)
print('HF repo:', f'https://huggingface.co/{HF_REPO}')
print('>>> Use this HF_REPO in inference_colab.ipynb:', HF_REPO)
def pull_ckpt(sub):
    try:
        snapshot_download(repo_id=HF_REPO, allow_patterns=[f'{sub}/*'], local_dir='.')
        print(f'[pull] {sub}: ' + ('resumable' if os.path.exists(f'{sub}/train_state.pt') else 'fresh'))
    except Exception as e:
        print(f'[pull] {sub}: nothing to resume ({e})')
def push_file(p):
    try: api.upload_file(path_or_fileobj=p, path_in_repo=os.path.basename(p), repo_id=HF_REPO,
                         commit_message=f'upload {p}'); print('[hf] pushed', p)
    except Exception as e: print('[hf] push skipped:', e)


## 2.5 · Stage 0 — build the corpus (fast, CPU)
Joins traces + answer-keys, makes the executor target `reasoning prose + FINAL ANSWER: <canonical>`,
and writes the **factored** `plan_vocab.json` (primitives + parameter-atoms + END). **Train = sets
1000+2000 (shuffled); held-out generalization = set 3000.** Seconds, no GPU.

In [ ]:
import json
TR = 'traces'; DATA = 'dataset/traces_sft.jsonl'
# vocab from ALL three sets so the planner head covers every token (incl. eval-only ones):
!python -u traces_to_sft.py --traces $TR/hard_reasoning_traces_1000.jsonl $TR/hard_reasoning_traces_2000.jsonl $TR/hard_reasoning_traces_3000.jsonl --answers $TR/answers_1000.jsonl $TR/answers_2000.jsonl $TR/answers_3000.jsonl --out /tmp/_all.jsonl --vocab_out plan_vocab.json
# TRAIN corpus = sets 1000 + 2000 (shuffled together):
!python -u traces_to_sft.py --traces $TR/hard_reasoning_traces_1000.jsonl $TR/hard_reasoning_traces_2000.jsonl --answers $TR/answers_1000.jsonl $TR/answers_2000.jsonl --out $DATA --vocab_out /tmp/_v.json
# HELD-OUT generalization = set 3000 (flipped-answer), never trained:
!python -u traces_to_sft.py --traces $TR/hard_reasoning_traces_3000.jsonl --answers $TR/answers_3000.jsonl --out dataset/traces_sft_3000.jsonl --vocab_out /tmp/_v.json
push_file('plan_vocab.json')
N = sum(1 for _ in open(DATA))
HELD = 60; TRAIN = min(N - HELD, 800); ROLL_TRAIN = min(TRAIN, 150); MAX_RESP = 256
print('planner vocab tokens:', len(json.load(open('plan_vocab.json'))['vocab']))
print(f'train corpus (1000+2000) rows={N}  ->  TRAIN={TRAIN}  HELD={HELD}  ROLL_TRAIN={ROLL_TRAIN}  MAX_RESP={MAX_RESP}')


## 3 · Stage 1 — Joint SFT  *(checkpoints + resumes via HF)*
Factored plan, `plan_max_len=24`, full-prose target (`max_resp=256`, `bs=2`). Watch the held
`ablation_gap` — with parameterized (load-bearing) plans it should turn **positive**.

In [ ]:
pull_ckpt('joint_ckpt')
!python -u train_sft.py --data $DATA --train $TRAIN --held $HELD --epochs 4 --lr 2e-5 --bs 2 --max_resp $MAX_RESP --plan_max_len 24 --probe 12 --out joint_ckpt --device cuda --resume --hf_repo $HF_REPO --ckpt_every_min 10


## 3.5 · Loss table (per epoch)
Color-coded `sft_metrics.jsonl`: training/held CE (green = lower) and `held_ablation_gap`
(green = positive = plan load-bearing). Re-run anytime during/after SFT.

In [ ]:
import json, os, pandas as pd
from IPython.display import display
if os.path.exists('sft_metrics.jsonl'):
    df = pd.DataFrame([json.loads(l) for l in open('sft_metrics.jsonl')])
    cols = ['epoch','time_s','train_plan_ce','train_resp_ce','kl','held_plan_ce','held_resp_ce',
            'held_acc_plan','held_acc_noplan','held_ablation_gap']
    df = df[[x for x in cols if x in df.columns]].drop_duplicates('epoch', keep='last')
    sty = df.style.hide(axis='index').format(precision=3).set_caption('SFT — per-epoch loss & held metrics')
    try:
        sty = sty.background_gradient(subset=['held_resp_ce','held_plan_ce'], cmap='RdYlGn_r')
        sty = sty.background_gradient(subset=['held_ablation_gap'], cmap='RdYlGn')
    except Exception: pass
    display(sty)
else:
    print('no sft_metrics.jsonl yet — run Stage 1 first.')


## 4 · Stage 2a — Offline rollouts

In [ ]:
pull_ckpt('joint_ckpt')
!python -u rollout_offline.py --sft_ckpt joint_ckpt --data $DATA --train $ROLL_TRAIN --group 8 --temp 1.2 --top_p 0.98 --max_resp $MAX_RESP --out rl_rollouts.jsonl --device cuda
push_file('rl_rollouts.jsonl')


## 5 · Stage 2b — Off-policy GRPO  *(checkpoints + resumes via HF)*

In [ ]:
pull_ckpt('joint_ckpt'); pull_ckpt('rl_ckpt')
import os; assert os.path.exists('rl_rollouts.jsonl'), 'run Stage 2a first'
!python -u train_grpo_offline.py --rollouts rl_rollouts.jsonl --sft_ckpt joint_ckpt --data $DATA --out rl_ckpt --inner_epochs 3 --lr 1e-4 --clip_eps 0.2 --beta_plan 1.0 --beta_ce 0.1 --held 16 --max_resp $MAX_RESP --device cuda --resume --hf_repo $HF_REPO --ckpt_every_min 10
print('All checkpoints on HF:', f'https://huggingface.co/{HF_REPO}/tree/main')


## 6 · Compare (in-distribution: held-out slice of 1000+2000)
`--sample` required to see RL effects.

In [ ]:
!python -u compare.py --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt --data $DATA --train $TRAIN --held $HELD --max_resp $MAX_RESP --sample --device cuda


## 7 · Generalization — set 3000 (flipped-answer, NEVER trained)
The headline test: trained on 1000+2000, evaluated on 3000 where the correct answer **flips**.
If accuracy holds here, the model is **reasoning, not memorizing surface patterns**.

In [ ]:
print('================ SET 3000 — flipped-answer, unseen ================')
!python -u compare.py --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt --data dataset/traces_sft_3000.jsonl --train 0 --held 150 --max_resp $MAX_RESP --sample --device cuda


## 8 · Head-to-head on one hard prompt (SFT vs SFT+RL)

In [ ]:
import torch
from model_joint import JointModel, decode_plan
HARD_PROMPT = ('A snail is at the bottom of a 12-meter well. Each day it climbs 4 meters, '
               'but each night it slides back 3 meters. On which day does it first reach the top?')
def run(ckpt, label, n=2, temp=0.7):
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda'); m.eval()
    print(f'\n{"="*70}\n{label}  ({ckpt})\n{"="*70}')
    with torch.no_grad():
        for k in range(n):
            p_ids, p_attn = m.batch_prompts([HARD_PROMPT])
            plan = m.sample_plan(p_ids, p_attn, temp=temp, sample=True)
            gen = m.generate_answer(p_ids, p_attn, plan, temp=temp, sample=True, max_new_tokens=256)
            print(f'  [sample {k+1}] plan: {" ".join(decode_plan(plan[0]))}')
            print('   ', m.tok.decode(gen[0], skip_special_tokens=True).strip()[:600])
    del m; torch.cuda.empty_cache()
run('joint_ckpt', 'SFT only'); run('rl_ckpt', 'SFT + GRPO')
